# Week 3 & 4: Environment Setup & Baseline RAG Pipeline

This notebook covers the tasks outlined in the PROJECT_PLAN.md for Week 3 and Week 4.

## Week 3: Environment Setup

### 1. Install Core Libraries
Make sure your virtual environment is active.

In [ ]:
!pip install -r ../requirements.txt
!pip install llama-index-embeddings-huggingface llama-index-vector-stores-qdrant llama-index-llms-openai qdrant-client transformers torch spacy ragas fastapi mlflow

### 2. Set up Docker for Qdrant
Run the following command in your terminal from the root of the repository to start Qdrant:

```bash
docker-compose up -d
```

### 3. Prepare Sample Datasets and Test Documents
Let's create a sample test document for our baseline.

In [ ]:
import os

data_dir = "../data"
os.makedirs(data_dir, exist_ok=True)

sample_text = """
Retrieval-Augmented Generation (RAG) is a technique that enhances large language models (LLMs) by augmenting their prompts with retrieved context.
Traditional RAG pipelines commonly use fixed-size chunking and single-stage vector retrieval.
These approaches often fail when the answer requires context from multiple sections of a document.
Context-Augmented Agentic Chunking (CAAC) is an intelligent chunking method that uses semantic boundaries.
Dynamic-Relevant RAG (DR-RAG) is a two-stage retrieval pipeline that first retrieves anchor chunks and then mines additional supporting evidence.
"""

sample_file_path = os.path.join(data_dir, "sample_document.txt")
with open(sample_file_path, "w") as f:
    f.write(sample_text)

print(f"Sample document created at {sample_file_path}")


## Week 4: Baseline RAG Pipeline

### 1. Fixed-Size Chunking & Embeddings generation using BGE-M3

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
import qdrant_client

# 1. Load Document
documents = SimpleDirectoryReader(data_dir).load_data()

# 2. Fixed-size chunking
# Using fixed-size chunking as baseline
text_parser = SentenceSplitter(chunk_size=128, chunk_overlap=20)
nodes = text_parser.get_nodes_from_documents(documents)
print(f"Created {len(nodes)} chunks (nodes).")

# 3. Generate embeddings using BGE-M3
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-m3")

# Set the global settings
Settings.embed_model = embed_model
Settings.text_splitter = text_parser


### 2. Store Baseline Chunks in Qdrant & Implement Retrieval

In [ ]:
# Initialize Qdrant Client (Assuming Docker is running locally on port 6333)
client = qdrant_client.QdrantClient(host="localhost", port=6333)

# Create a local vector store (or connect to it)
vector_store = QdrantVectorStore(client=client, collection_name="baseline_rag")

# Setup Storage Context
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Build the Index (Stores baseline chunks in Qdrant)
index = VectorStoreIndex(nodes, storage_context=storage_context)
print("Chunks stored in Qdrant Vector Store.")


### 3. Single-stage top-k retrieval & Answer Generation
*(Note: For answer generation, we use a mock LLM or OpenAI. If you don't have an OpenAI API key, you can configure a local LLM instead.)*

In [ ]:
from llama_index.core.llms import MockLLM

# Using a Mock LLM for baseline pipeline completeness without API keys
Settings.llm = MockLLM(max_tokens=256)

# Implement single-stage top-k retrieval
query_engine = index.as_query_engine(similarity_top_k=2)

# Querying
query = "What is Context-Augmented Agentic Chunking?"
response = query_engine.query(query)

print(f"Query: {query}")
print(f"Generated Answer: {response}")
print("-" * 50)
print("Retrieved Source Chunks:")
for node in response.source_nodes:
    print(f"- {node.node.text}")
